In [ ]:
#  MANIPULATION DE DONNÉES
# ──────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────────────────
#  VISUALISATION
# ──────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#  PREPROCESSING
# ──────────────────────────────────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler, 
    MinMaxScaler,
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
# ──────────────────────────────────────────────────────────────────────────# 1.4 SÉPARATION DES DONNÉES
# ──────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV
)


# Top 5 Modèles
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


# Métriques
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)
import time

In [ ]:
# ==============================
# CHARGEMENT DES DONNÉES
# ==============================

data = pd.read_excel("/kaggle/input/datasets/sitaouedraogo/tp1-tinyml/ventilateur_maintenance.xlsx")

# Aperçu des premières lignes
print("Aperçu des données :")
print(data.head())

# Informations générales
print("\nInformations sur le dataset :")
print(data.info())

# Statistiques descriptives
print("\nStatistiques descriptives :")
print(data.describe())

# Vérification des valeurs manquantes
print("\nValeurs manquantes :")
print(data.isnull().sum())


In [ ]:
plt.figure()
sns.heatmap(data.corr(), annot=True)
plt.title("Matrice de corrélation")
plt.show()
plt.figure()
sns.countplot(x='Panne', data=data)
plt.title("Distribution des classes (Panne)")
plt.show()


In [ ]:
# Séparation X et y
print("1️⃣ Séparation des features et de la cible...")
X = data.drop('Panne', axis=1)
y = data['Panne']

print(f"   ✅ Features (X): {X.shape}")
print(f"   ✅ Target (y): {y.shape}")
print(f"   📋 Features: {list(X.columns)}")


#  Split Train/Test (80/20)
print("\n2️⃣ Séparation Train/Test (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Important pour garder les proportions
)

print(f"   ✅ Train: {X_train.shape[0]} échantillons ({X_train.shape[0]/len(data)*100:.1f}%)")
print(f"   ✅ Test:  {X_test.shape[0]} échantillons ({X_test.shape[0]/len(data)*100:.1f}%)")

print(f"\n   Distribution Train:")
print(f"   {y_train.value_counts()}")
print(f"\n   Distribution Test:")
print(f"   {y_test.value_counts()}")

# 4.3 Normalisation (IMPORTANT: APRÈS le split!)
print("\n3️⃣ Normalisation des features...")
print("   Méthode: StandardScaler (mean=0, std=1)")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit sur train seulement!
X_test_scaled = scaler.transform(X_test)        # Transform sur test

# Reconvertir en DataFrame pour garder les noms de colonnes
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print(f"   ✅ Features normalisées")
print(f"   📊 Vérification - Moyenne train: {X_train_scaled.mean().mean():.6f}")
print(f"   📊 Vérification - Std train: {X_train_scaled.std().mean():.6f}")

# 4.4 Équilibrage des classes avec SMOTE
print("\n4️⃣ Équilibrage des classes avec SMOTE...")
print(f"   Distribution AVANT SMOTE:")
print(f"   {y_train.value_counts()}")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"\n   Distribution APRÈS SMOTE:")
print(f"   {pd.Series(y_train_balanced).value_counts()}")
print(f"   ✅ Classes équilibrées!")

print("\n" + "="*70)
print("✅ PREPROCESSING TERMINÉ!")
print("="*70 + "\n")


In [ ]:

print("="*70)
print("🚀 ENTRAÎNEMENT DES TOP 5 MODÈLES")
print("="*70 + "\n")

# Définir les modèles
models = {
    '1. XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ),
    
    '2. LightGBM': LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    ),
    
    '3. CatBoost': CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        random_state=42,
        verbose=False
    ),
    
    '4. Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    
    '5. Neural Network': MLPClassifier(
        hidden_layer_sizes=(100, 50, 25),
        max_iter=500,
        learning_rate_init=0.001,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
}

# Stocker les résultats
results = {}

# Entraîner chaque modèle
for name, model in models.items():
    print(f"{'─'*70}")
    print(f"🔄 Entraînement: {name}")
    print(f"{'─'*70}")
    
    # Timer
    start_time = time.time()
    
    # Entraînement
    model.fit(X_train_balanced, y_train_balanced)
    
    # Prédictions sur test (données NON équilibrées)
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Temps d'entraînement
    training_time = time.time() - start_time
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    
    # Stocker les résultats
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'training_time': training_time,
        'predictions': y_pred,
        'probabilities': y_pred_proba,
        'confusion_matrix': cm
    }
    
    # Afficher les résultats
    print(f"✅ Résultats:")
    print(f"   • Accuracy:  {accuracy:.4f}")
    print(f"   • Precision: {precision:.4f}")
    print(f"   • Recall:    {recall:.4f}")
    print(f"   • F1-Score:  {f1:.4f}")
    print(f"   • ROC-AUC:   {roc_auc:.4f}")
    print(f"   • Temps:     {training_time:.2f}s")
    print()

print("="*70)
print("✅ TOUS LES MODÈLES ENTRAÎNÉS!")
print("="*70 + "\n")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  COMPARAISON DES MODÈLES
# ═══════════════════════════════════════════════════════════════════════════

print("="*70)
print("📊 COMPARAISON DES MODÈLES")
print("="*70 + "\n")

# Créer le DataFrame de comparaison
comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Modèle': name,
        'Accuracy': res['accuracy'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'F1-Score': res['f1_score'],
        'ROC-AUC': res['roc_auc'],
        'Temps (s)': res['training_time']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

# Afficher le tableau
print("📋 TABLEAU DE COMPARAISON:")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70 + "\n")

# Identifier le meilleur modèle
best_model_name = comparison_df.iloc[0]['Modèle']
best_f1 = comparison_df.iloc[0]['F1-Score']
best_accuracy = comparison_df.iloc[0]['Accuracy']

print(f"🏆 MEILLEUR MODÈLE: {best_model_name}")
print(f"   • F1-Score: {best_f1:.4f}")
print(f"   • Accuracy: {best_accuracy:.4f}")
print()


In [ ]:
# Visualisation - Barplots des métriques
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Temps (s)']
colors = ['skyblue', 'lightcoral', 'lightgreen', 'gold', 'plum', 'lightsalmon']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    row = idx // 3
    col = idx % 3
    
    ax = axes[row, col]
    comparison_df.plot(x='Modèle', y=metric, kind='barh', ax=ax, 
                       color=color, legend=False)
    ax.set_title(f'{metric}', fontsize=14, fontweight='bold')
    ax.set_xlabel(metric)
    ax.set_ylabel('')
    
    # Ajouter les valeurs
    for i, v in enumerate(comparison_df[metric]):
        ax.text(v, i, f' {v:.3f}', va='center')

plt.tight_layout()

In [ ]:
# Heatmap des métriques
print("📊 Heatmap des performances:\n")

metrics_df = comparison_df.set_index('Modèle')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]

plt.figure(figsize=(12, 6))
sns.heatmap(metrics_df.T, annot=True, fmt='.3f', cmap='RdYlGn', 
            center=0.5, linewidths=1, cbar_kws={'label': 'Score'})
plt.title('Heatmap des Métriques - Tous les Modèles', fontsize=16, fontweight='bold')
plt.xlabel('Modèle')
plt.ylabel('Métrique')
plt.tight_layout()

In [ ]:
#  Matrices de confusion
print("🔲 Matrices de confusion:\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    cm = res['confusion_matrix']
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                cbar=False, square=True, 
                xticklabels=['Pas de Panne', 'Panne'],
                yticklabels=['Pas de Panne', 'Panne'])
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Prédictions')
    axes[idx].set_ylabel('Vraies Valeurs')
    
    # Ajouter les pourcentages
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            pct = 100 * cm[i, j] / total
            axes[idx].text(j + 0.5, i + 0.7, f'({pct:.1f}%)', 
                          ha='center', va='center', fontsize=9, color='gray')

# Masquer le dernier axe
axes[-1].axis('off')

plt.tight_layout()
plt.savefig('matrices_confusion.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Courbes ROC
print("📈 Courbes ROC:\n")

plt.figure(figsize=(12, 8))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['probabilities'])
    roc_auc_value = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, lw=2, 
             label=f'{name} (AUC = {roc_auc_value:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Aléatoire (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
plt.title('Courbes ROC - Comparaison des Modèles', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()